# Taskforce Playground

Interaktives Notebook zum Spielen mit Taskforce-Agenten und gleichzeitig eine Tour
durch die programmatische API:

1. **Setup & Smoke-Test** – Imports, Environment, LLM-Verbindung
2. **Minimal-Agent** – `AgentFactory.create_agent(...)` mit Inline-Parametern
3. **Mission ausführen** – `AgentExecutor` mit Streaming-Events
4. **Tool-Auswahl** – verfügbare Tools auflisten, Subset wählen
5. **Planning-Strategien** – `native_react`, `plan_and_execute`, `plan_and_react`, `spar`
6. **Context-Engineering** – `ContextPolicy`, Message-Caps, Tool-Result-Store
7. **Profile (YAML)** – Agent aus Profil laden
8. **Custom Tool** – eigenes `BaseTool` schreiben und einhängen
9. **Aufräumen** – Agent sauber schließen (MCP-Connections etc.)

Voraussetzungen:
- `uv sync` im Repo-Root ausgeführt
- LLM-Credentials gesetzt (z. B. `AZURE_API_KEY`, `AZURE_API_BASE`, `AZURE_API_VERSION` oder `OPENAI_API_KEY`)
- Notebook im Repo-Root starten, damit relative Pfade (`src/taskforce/configs/...`) auflösen


## 1. Setup & Smoke-Test

Wir sorgen dafür, dass das Repo im `sys.path` ist (falls das Notebook außerhalb von `uv run` läuft)
und prüfen kurz die Umgebung. `nest_asyncio` brauchen wir, weil Jupyter bereits einen Event-Loop hat.


In [1]:
import os
import sys
from pathlib import Path

# Repo-Root sicherstellen (wir nehmen an, das Notebook liegt unter notebooks/)
REPO_ROOT = Path.cwd()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
os.chdir(REPO_ROOT)

src_path = str(REPO_ROOT / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

print("cwd:", REPO_ROOT)
print("python:", sys.version.split()[0])

# Optional: .env laden, falls vorhanden
try:
    from dotenv import load_dotenv
    if (REPO_ROOT / ".env").exists():
        load_dotenv(REPO_ROOT / ".env")
        print(".env geladen")
except ImportError:
    pass

# Welche LLM-Provider-Keys sind gesetzt? (nur Existenz, kein Leak)
for k in ("OPENAI_API_KEY", "ANTHROPIC_API_KEY", "AZURE_API_KEY", "AZURE_API_BASE", "GEMINI_API_KEY"):
    print(f"  {k:24s} {'OK' if os.getenv(k) else '-'}")


cwd: c:\Users\rudi\source\pytaskforce
python: 3.11.1
.env geladen
  OPENAI_API_KEY           OK
  ANTHROPIC_API_KEY        OK
  AZURE_API_KEY            OK
  AZURE_API_BASE           OK
  GEMINI_API_KEY           -


In [2]:
# IPython unterstützt top-level await out-of-the-box. nest_asyncio nur,
# falls man später asyncio.run(...) o.Ä. parallel verwenden will.
try:
    import nest_asyncio
    nest_asyncio.apply()
except ImportError:
    pass

import asyncio
import structlog

# Logging hübscher machen; auf WARNING setzen, sonst wird das Notebook geflutet.
import logging
logging.basicConfig(level=logging.WARNING, format="%(levelname)s %(name)s | %(message)s")
structlog.configure(wrapper_class=structlog.make_filtering_bound_logger(logging.WARNING))


## 2. Minimal-Agent

Der einfachste Weg: `AgentFactory.create_agent(...)` mit Inline-Parametern. **Wichtig:**
Profile/Config-Datei und Inline-Parameter sind **mutually exclusive** – entweder/oder.

Wir bauen einen Agenten mit nur zwei Tools (`python`, `web_search`) und einem eigenen
System-Prompt.


In [3]:
from taskforce.application.factory import AgentFactory

factory = AgentFactory()

agent = await factory.create_agent(
    system_prompt=(
        "Du bist ein knapper, freundlicher Assistent. "
        "Antworte standardmäßig auf Deutsch."
    ),
    tools=["python", "web_search"],
    persistence={"type": "file", "work_dir": ".taskforce_playground"},
    max_steps=8,
)

print("Agent OK")
print("  Tools     :", [t for t in agent.tools])
print("  Max steps :", agent.max_steps)
print("  Model     :", agent.model_alias)
print("  Strategy  :", agent.planning_strategy.name)


Agent OK
  Tools     : ['python', 'web_search', 'planner']
  Max steps : 8
  Model     : main
  Strategy  : native_react


## 3. Mission ausführen (mit Streaming)

Statt `agent.execute(...)` direkt aufzurufen, nutzen wir den `AgentExecutor` aus dem
Application-Layer – das ist derselbe Pfad, den CLI und REST-API verwenden. Er liefert
ProgressUpdate-Events, die wir live mitlesen.


In [ ]:
import json

from taskforce.application.executor import AgentExecutor
from taskforce.core.domain.enums import EventType

executor = AgentExecutor(factory=factory)

INTERESTING = {
    EventType.STEP_START.value,
    EventType.PLAN_UPDATED.value,
    EventType.TOOL_CALL.value,
    EventType.TOOL_RESULT.value,
    EventType.FINAL_ANSWER.value,
    EventType.COMPLETE.value,
    EventType.ERROR.value,
}


def full_event_message(ev) -> str:
    """Return the most complete human-readable message available on a ProgressUpdate."""
    details = ev.details or {}

    if ev.event_type_value == EventType.TOOL_CALL.value:
        args = details.get("args")
        if args:
            return (
                f"Calling: {details.get('tool', 'unknown')}\n"
                f"args: {json.dumps(args, ensure_ascii=False, indent=2, default=str)}"
            )
        return ev.message or ""

    if ev.event_type_value == EventType.TOOL_RESULT.value:
        status = "OK" if details.get("success") else "FAIL"
        output = details.get("output", "")
        if not isinstance(output, str):
            output = json.dumps(output, ensure_ascii=False, indent=2, default=str)
        return f"{status} {details.get('tool', 'unknown')}:\n{output}"

    if ev.event_type_value == EventType.PLAN_UPDATED.value:
        action = details.get("action", "unknown")
        plan = details.get("plan")
        if plan:
            return f"{action}:\n{plan}"

        steps = details.get("steps")
        if steps:
            rendered_steps = "\n".join(
                f"[ ] {index}. {step}" for index, step in enumerate(steps, 1)
            )
            return f"{action}:\n{rendered_steps}"

        return ev.message or ""

    return (ev.message or "").strip()


# WICHTIG:
# - Wir reichen den Agent als `agent=` herein, damit der Executor keinen neuen baut.
# - `profile="default"` (statt des Default "butler") sorgt dafür, dass das
#   Post-Mission-Learning nach Mission-Ende ein existierendes Profil findet
#   (auf Setups ohne `taskforce-butler`-Package).
# - Wir BREAKEN NICHT aus dem `async for`. Vorzeitiges Schließen eines Async-
#   Generators in Jupyter löst `GeneratorExit` in einem anderen asyncio-
#   Context aus -> der ContextVar-Token des Executors lässt sich dann nicht
#   resetten ("Token was created in a different Context"). Stattdessen kappen
#   wir nur die Ausgabe und lassen den Generator zu Ende laufen.
async def run(mission: str, *, agent=agent, max_events: int = 60):
    shown = 0
    truncated = False
    async for ev in executor.execute_mission_streaming(
        mission=mission, agent=agent, profile="default"
    ):
        if ev.event_type_value not in INTERESTING:
            continue
        if shown < max_events:
            msg = full_event_message(ev)
            print(f"[{ev.event_type_value:14s}] {msg}")
            shown += 1

        elif not truncated:
            print("... (Ausgabe gekappt; Mission läuft im Hintergrund weiter)")
            truncated = True

await run("Berechne mit dem python-Tool 17 * 23 und nenne mir das Ergebnis.")


[tool_call     ] Calling: python
args: {
  "code": "result = 17 * 23"
}
[tool_result   ] OK python:
{"success": true, "result": 391, "variables": ["tool_web_search", "result"]}
[final_answer  ] 17 × 23 = 391
[complete      ] Execution completed. Status: completed


## 4. Tool-Auswahl

Die `ToolRegistry` ist die Single-Source-of-Truth für Tool-Kurznamen. Wir listen sie auf
und schauen uns Detail-Specs an. Die im Profil/Inline-Param übergebenen Namen werden
darüber aufgelöst.


In [5]:
from taskforce.application.tool_registry import ToolRegistry
from taskforce.infrastructure.tools.registry import get_all_tool_names, get_tool_definition

all_names = sorted(get_all_tool_names())
print(f"Insgesamt {len(all_names)} Tools registriert:\n")
for name in all_names:
    spec = get_tool_definition(name)
    print(f"  {name:28s} -> {spec['type']:28s} ({spec['module']})")


Insgesamt 39 Tools registriert:

  accounting_audit             -> AccountingAuditTool          (taskforce.infrastructure.tools.native.accounting_audit_tool)
  accounting_validate          -> AccountingValidateTool       (taskforce.infrastructure.tools.native.accounting_validate_tool)
  activate_skill               -> ActivateSkillTool            (taskforce.infrastructure.tools.native.activate_skill_tool)
  ask_user                     -> AskUserTool                  (taskforce.infrastructure.tools.native.ask_user_tool)
  authenticate                 -> AuthTool                     (taskforce.infrastructure.tools.native.auth_tool)
  bash                         -> BashTool                     (taskforce.infrastructure.tools.native.shell_tool)
  browser                      -> BrowserTool                  (taskforce.infrastructure.tools.native.browser_tool)
  calendar                     -> CalendarTool                 (taskforce_google_workspace.calendar)
  call_acp_agent              

In [6]:
# Tools instantiieren ohne kompletten Agent zu bauen — gut zum Inspizieren.
registry = ToolRegistry(llm_provider=None)
tools = registry.resolve(["python", "web_search", "file_read"])
for t in tools:
    print(f"\n=== {t.name} ===")
    print("  Beschreibung :", (t.description or "")[:120])
    print("  Approval?    :", getattr(t, "requires_approval", False))
    print("  Parameters   :", list((t.parameters_schema.get("properties") or {}).keys()))



=== python ===
  Beschreibung : Execute Python code for complex logic, data processing, and custom operations. Your code must assign the final output to
  Approval?    : True
  Parameters   : ['code', 'context', 'cwd']

=== web_search ===
  Beschreibung : Search the web using DuckDuckGo
  Approval?    : False
  Parameters   : ['query', 'num_results', 'snippet_max_chars']

=== file_read ===
  Beschreibung : Read file contents safely with size limits and encoding detection
  Approval?    : False
  Parameters   : ['path', 'encoding', 'max_size_mb']


## 5. Planning-Strategien

Vier Strategien stehen zur Verfügung — jede hat ein anderes Trade-off zwischen
Vorausplanung und Reaktivität:

| Strategie | Verhalten |
|-----------|-----------|
| `native_react` | Default – reine Thought→Action→Observation-Schleife |
| `plan_and_execute` | Plant zuerst, führt Schritte sequentiell aus |
| `plan_and_react` | Hybrid – Plan auf hohem Level, ReAct pro Step |
| `spar` | Sense → Plan → Act → Reflect mit iterativem Refining |

Die `planning_strategy_params` hängen von der Strategie ab — siehe
`taskforce/application/planning_strategy_factory.py` für die akzeptierten Keys.


In [7]:
from taskforce.application.planning_strategy_factory import select_planning_strategy

for name, params in [
    ("native_react", None),
    ("plan_and_execute", {"max_step_iterations": 4, "max_plan_steps": 8}),
    ("plan_and_react", {"max_plan_steps": 6}),
    ("spar", {"max_step_iterations": 3, "reflect_every_step": True, "max_reflection_iterations": 2}),
]:
    strategy = select_planning_strategy(name, params)
    print(f"  {name:18s} -> {type(strategy).__name__}")


  native_react       -> NativeReActStrategy
  plan_and_execute   -> PlanAndExecuteStrategy
  plan_and_react     -> PlanAndReactStrategy
  spar               -> SparStrategy


In [14]:
# Agent mit Plan-and-Execute bauen und eine mehrstufige Mission ausführen.
planner = await factory.create_agent(
    system_prompt="Plane zuerst auf Deutsch, führe dann aus. Halte dich kurz.",
    tools=["python"],
    persistence={"type": "file", "work_dir": ".taskforce_playground"},
    planning_strategy="plan_and_execute",
    planning_strategy_params={"max_step_iterations": 3, "max_plan_steps": 6},
    max_steps=15,
)

await run(
    "Berechne die Summe der Quadrate von 1 bis 10 mit dem python-Tool "
    "und vergleiche das Ergebnis mit der Formel n*(n+1)*(2n+1)/6.",
    agent=planner,
    max_events=80,
)

await planner.close()


[plan_updated  ] Plan updated (create_plan)
[tool_call     ] Calling: python
args: {
  "code": "result = sum(i*i for i in range(1, 11))"
}
[tool_result   ] OK python:
{"success": true, "result": 385, "variables": ["result"]}
[tool_call     ] Calling: planner
args: {
  "action": "mark_done",
  "step_index": 1
}
[tool_result   ] OK planner:
[x] 1. Mit dem Python-Tool die Summe der Quadrate von 1 bis 10 berechnen.
[ ] 2. Mit dem Python-Tool die Formel n*(n+1)*(2n+1)/6 für n=10 auswerten.
[ ] 3. Die beiden Ergebnisse vergleichen und das Ergebnis kurz zusammenfassen.
[plan_updated  ] Plan updated (mark_done)
[tool_call     ] Calling: python
args: {
  "code": "n = 10\nresult = n*(n+1)*(2*n+1)//6"
}
[tool_result   ] OK python:
{"success": true, "result": 385, "variables": ["n", "result"]}
[tool_call     ] Calling: planner
args: {
  "action": "mark_done",
  "step_index": 2
}
[tool_result   ] OK planner:
[x] 1. Mit dem Python-Tool die Summe der Quadrate von 1 bis 10 berechnen.
[x] 2. Mit dem Py

## 6. Context-Engineering

Drei Hebel kontrollieren, was vor jedem LLM-Call im Context landet:

- **`ContextPolicy`** — Hard-Caps für den Context-Pack-Builder (Items, Chars, Tool-Previews).
- **`tool_result_store_threshold`** — ab welcher Größe Tool-Resultate auf Disk gelegt
  und im Chat-Log nur als Referenz geführt werden (ADR-025).
- **`tool_message_max_chars` / `assistant_message_max_chars`** — Hard-Cap pro
  Message-Rolle.

Mit kleinen Werten zwingen wir den Agenten, sparsam mit Tokens umzugehen.


In [9]:
from taskforce.core.domain.context_policy import ContextPolicy

tight_policy = ContextPolicy(
    max_items=4,
    max_chars_per_item=200,
    max_total_chars=600,
    include_latest_tool_previews_n=2,
)
print("Tight policy:", tight_policy.to_dict())

tight_agent = await factory.create_agent(
    system_prompt="Antworte kurz auf Deutsch.",
    tools=["python", "web_search"],
    persistence={"type": "file", "work_dir": ".taskforce_playground"},
    context_policy=tight_policy.to_dict(),
    max_steps=6,
)
print("Context policy am Agent:", tight_agent.context_policy.to_dict())


Tight policy: {'max_items': 4, 'max_chars_per_item': 200, 'max_total_chars': 600, 'include_latest_tool_previews_n': 2, 'allow_tools': None, 'allow_selectors': ['first_chars'], 'deduplicate_visible_window': 10}
Context policy am Agent: {'max_items': 4, 'max_chars_per_item': 200, 'max_total_chars': 600, 'include_latest_tool_previews_n': 2, 'allow_tools': None, 'allow_selectors': ['first_chars'], 'deduplicate_visible_window': 10}


In [10]:
# Context-Snapshot inspizieren — zeigt Messages, Tools und (optional) Sub-Agent-Einträge.
snap = tight_agent.context.snapshot(include_content=True)
print(f"Messages im Context : {len(snap.messages)}")
print(f"Tools im Context    : {len(snap.tools)}")
for m in snap.messages[:4]:
    content = str(m.get('content', ''))[:120]
    print(f"  - role={m['role']:9s} chars={len(str(m.get('content','')))}: {content}...")


Messages im Context : 0
Tools im Context    : 3


## 7. Profil (YAML) laden

Statt jedes Mal Inline-Parameter zu setzen, kann man ein YAML-Profil laden. Das
Default-Profil liegt in `src/taskforce/configs/default.yaml`. Eigene Profile passen
ins selbe Verzeichnis oder werden per absolutem Pfad übergeben.


In [12]:
# Den Default-Profil-Agent bauen (komplettes Tool-Set: file_read, browser, git, wiki, ...)
default_agent = await factory.create_agent(profile="default")
print("Default-Agent OK")
print("  Tools    :", [t for t in default_agent.tools])
print("  Strategy :", default_agent.planning_strategy.name)
print("  Max steps:", default_agent.max_steps)
await default_agent.close()


Default-Agent OK
  Tools    : ['web_search', 'web_fetch', 'browser', 'file_read', 'file_write', 'edit', 'multimedia', 'python', 'powershell', 'grep', 'glob', 'git', 'ask_user', 'wiki', 'planner']
  Strategy : native_react
  Max steps: 120


## 8. Custom Tool schreiben

Eigene Tools erbt man am einfachsten von `BaseTool` — das spart Boilerplate. Hier ein
kleines Würfel-Tool, das wir direkt in die Tool-Liste eines Agenten injizieren
(ohne Registry-Eintrag — nützlich für Notebook-Experimente).


In [ ]:
import random
from typing import Any
from taskforce.infrastructure.tools.base_tool import BaseTool

class DiceRollTool(BaseTool):
    tool_name = "dice_roll"
    tool_description = "Wirft einen N-seitigen Würfel und liefert das Ergebnis."
    tool_parameters_schema = {
        "type": "object",
        "properties": {
            "sides": {"type": "integer", "description": "Anzahl Seiten (>=2)", "default": 6},
            "count": {"type": "integer", "description": "Anzahl Würfe", "default": 1},
        },
        "required": [],
    }

    async def _execute(self, **params: Any) -> dict[str, Any]:
        sides = int(params.get("sides", 6))
        count = int(params.get("count", 1))
        if sides < 2 or count < 1:
            return {"success": False, "error": "sides>=2 und count>=1 erforderlich"}
        rolls = [random.randint(1, sides) for _ in range(count)]
        return {"success": True, "output": {"rolls": rolls, "sum": sum(rolls)}}

# Schnelltest ohne Agent
dice = DiceRollTool()
print(await dice.execute(sides=20, count=3))


In [ ]:
# Agent mit Custom-Tool: wir bauen einen Standard-Agent und hängen den Würfel danach an.
dice_agent = await factory.create_agent(
    system_prompt=(
        "Du hast ein Tool 'dice_roll'. Nutze es, um Würfelfragen zu beantworten. "
        "Antworte knapp auf Deutsch."
    ),
    tools=[],  # leer — wir injizieren gleich manuell
    persistence={"type": "file", "work_dir": ".taskforce_playground"},
    max_steps=6,
)
dice_agent.tools.append(DiceRollTool())
# Den System-Prompt nicht neu zusammenbauen — der Inline-Prompt nennt das Tool bereits.

await run(
    "Würfle dreimal mit einem zwanzigseitigen Würfel und sag mir die Summe.",
    agent=dice_agent,
)
await dice_agent.close()


### Custom Tool dauerhaft registrieren (optional)

Wenn das Tool dauerhaft per Kurzname (`dice_roll` in einer Profile-YAML) erreichbar
sein soll, gibt es zwei Wege:

1. **Eintrag in `_BUILTIN_REGISTRY`** in `src/taskforce/infrastructure/tools/registry.py`
   (für Tools, die ins Framework gehören).
2. **Entry-Point** in einem eigenen Agent-Package unter der Gruppe `taskforce.tools`
   (siehe ADR-026 und CLAUDE.md → "Unified CLI").

Beides ist nichts für Notebook-Experimente, sondern für richtige Projekte.


## 9. Aufräumen

Wenn ein Agent MCP-Connections oder Plugin-Tools geöffnet hat, **muss** man `close()`
aufrufen, sonst bleiben Subprozesse/Locks hängen. In diesem Notebook gibt es keine
MCP-Server, aber wir gewöhnen es uns trotzdem an.


In [ ]:
for a in (agent, tight_agent):
    try:
        await a.close()
    except Exception as e:
        print("close-Fehler:", e)
print("fertig.")


## Weiterführend

- Dynamisches LLM-Routing (verschiedene Modelle pro Phase): `src/taskforce/configs/llm_config.yaml`
  → `routing:` Block. ADR-012.
- Sub-Agents / Specialists: `factory.create_agent(specialist="research_specialist", ...)`.
- Wiki-Memory (`wiki`-Tool): markdown-basierte Long-Term-Memory, ADR-020.
- MCP-Server anhängen: `mcp_servers=[{...}]` Parameter an `create_agent(...)`.
- Tool-Result-Store anschauen: `.taskforce_playground/tool_results/`.
- Vollständige Architektur-Doku: `CLAUDE.md` im Repo-Root.
